# Verify the PyTorch projection correlation implementation

This notebook expects the project root to be the working directory, with `projcorr.py`, `pcor_ipcor.py` and both CSV files in the same folder. In Colab, it automatically selects CUDA when a GPU is available.

Timing is reported for every tested procedure.

In [1]:
import os
import sys
import time
import json
import glob
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

if 'google.colab' in sys.modules:
    print("[Colab] Google Colab environment detected.")
    from google.colab import drive
    drive.mount('/content/drive')

    candidates = [
        '/content/drive/MyDrive/user/goit/Diplom/var1_Di/code/projCorr',
        '/content/drive/MyDrive/Diplom/var1_Di/code/projCorr',
        '/content/drive/MyDrive/var1_Di/code/projCorr',
        '/content/drive/MyDrive/code/projCorr'
    ]
    target_dir = None
    for cand in candidates:
        if os.path.isdir(cand):
            target_dir = cand
            break
    if target_dir:
        os.chdir(target_dir)
        if target_dir not in sys.path:
            sys.path.insert(0, target_dir)
        print(f"[Colab] Working directory set to: {os.getcwd()}")
else:
    current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
    if current_dir not in sys.path:
        sys.path.insert(0, current_dir)
    print(f"[Local] Working directory: {os.getcwd()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


[Colab] Google Colab environment detected.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Colab] Working directory set to: /content/drive/MyDrive/user/goit/Diplom/var1_Di/code/projCorr
Device: cuda
GPU: Tesla T4
VRAM: 15.64 GB


In [2]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import torch
from projcorr import pcor_test, ipcor_test, ipcor_rcv_test
from pcor_ipcor import pcor, ipcor

x_path = Path('data_20261109_195329_x.csv')
y_path = Path('data_20261109_195329_y.csv')
assert Path('projcorr.py').exists(), f'projcorr.py was not found in {Path.cwd()}'
assert Path('pcor_ipcor.py').exists(), f'pcor_ipcor.py was not found in {Path.cwd()}'
assert x_path.exists() and y_path.exists(), f'Missing data files: {x_path}, {y_path}'

x_df = pd.read_csv(x_path)
y_df = pd.read_csv(y_path)
x = x_df.values.astype(np.float64)
y = y_df.values.astype(np.float64)
n = len(x)

timing_results = []

def timed(name, fn):
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    timing_results.append((name, elapsed))
    print(f"  [timed] {name}: {elapsed:.4f} s")
    return result

print(f"Working directory: {os.getcwd()}")
print(f"Compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Data shapes: x={x.shape}, y={y.shape}")


Working directory: /content/drive/MyDrive/user/goit/Diplom/var1_Di/code/projCorr
Compute device: cuda
GPU: Tesla T4
Data shapes: x=(100, 5), y=(100, 5)


## 1. `pcor_test`

Expected values: T = 0.27747, PC = 0.4889854, p = 0.001.

In [3]:
def run_pcor_test():
    res = pcor_test(x, y, n_perm=999, seed=1, device=device)
    print(res)
    assert abs(res.statistic['T'] - 0.27747) < 1e-4, f"Mismatch in T: {res.statistic['T']}"
    assert abs(res.estimate['PC'] - 0.4889854) < 1e-5, f"Mismatch in PC: {res.estimate['PC']}"
    assert abs(res.p_value - 0.001) < 1e-3, f"Mismatch in p-value: {res.p_value}"
    print('pcor_test matches the reference.')
    return res

res_pcor = timed('pcor_test', run_pcor_test)


	Projection correlation permutation test

data:  x and y
T = 0.27747, permutations = 999, p-value = 0.001
alternative hypothesis: true correlation is greater than 0
sample estimates:
          PC
   0.4889854

pcor_test matches the reference.
  [timed] pcor_test: 0.8563 s


## 1b. `pcor` (from `pcor_ipcor`)

Pure scalar Projection Correlation without hypothesis test. Returns a float in [0, 1].

In [4]:
def run_pcor_pure():
    val = pcor(x, y, chunk_size=64, device=device)
    print(f"pcor = {val:.6f}")
    assert 0.0 <= val <= 1.0, f"pcor out of range: {val}"
    print('pcor (pcor_ipcor) executed successfully.')
    return val

pcor_val = timed('pcor (pcor_ipcor)', run_pcor_pure)

pcor = 0.488985
pcor (pcor_ipcor) executed successfully.
  [timed] pcor (pcor_ipcor): 0.1063 s


## 2. `ipcor_test`

Expected values: Z = 12.093, IPC = 0.4135423, p < 2.2e-16.

In [5]:
def run_ipcor_test():
    res = ipcor_test(x, y, device=device)
    print(res)
    assert abs(res.statistic['Z'] - 12.093) < 1e-3, f"Mismatch in Z: {res.statistic['Z']}"
    assert abs(res.estimate['IPC'] - 0.4135423) < 1e-5, f"Mismatch in IPC: {res.estimate['IPC']}"
    assert res.p_value < 1e-15, f"Mismatch in p-value: {res.p_value}"
    print('ipcor_test matches the reference.')
    return res

res_ipcor = timed('ipcor_test', run_ipcor_test)


	Improved projection correlation asymptotic test

data:  x and y
Z = 12.093, p-value = < 2.2e-16
alternative hypothesis: true correlation is greater than 0
sample estimates:
         IPC
   0.4135423

ipcor_test matches the reference.
  [timed] ipcor_test: 0.0346 s


## 2b. `ipcor` (from `pcor_ipcor`)

Pure scalar Improved Projection Correlation without hypothesis test. Returns a float in [0, 1].

In [9]:
def run_ipcor_pure():
    val = ipcor(x, y, device=device)
    print(f"ipcor = {val:.7f}")
    assert 0.0 <= val <= 1.0, f"ipcor out of range: {val}"
    print('ipcor (pcor_ipcor) executed successfully.')
    return val

ipcor_val = timed('ipcor (pcor_ipcor)', run_ipcor_pure)

ipcor = 0.4135423
ipcor (pcor_ipcor) executed successfully.
  [timed] ipcor (pcor_ipcor): 0.0070 s


## 3. `ipcor_rcv_test`

Set the number of pairs to `ceil(log(n))`. To reproduce the reference R split exactly, pass the same `first_half` indices used in `test_projcorr.py`. Expected values: Z = 13.278, RCV T = 0.2655577.

In [7]:
def run_ipcor_rcv_test():
    first_half_r = [0, 1, 5, 6, 13, 17, 19, 20, 21, 23, 24, 27, 31, 32, 33, 34, 36, 37, 38, 39, 41, 42, 43, 44, 50, 53, 58, 63, 67, 68, 69, 71, 72, 73, 74, 78, 79, 81, 82, 83, 84, 86, 89, 90, 91, 93, 95, 96, 97, 98]
    n_pairs = math.ceil(math.log(n))
    res = ipcor_rcv_test(x, y, n_pairs=n_pairs, seed=1, first_half=first_half_r, device=device)
    print(res)
    assert abs(res.statistic['Z'] - 13.278) < 1e-3, f"Mismatch in Z: {res.statistic['Z']}"
    assert abs(res.estimate['RCV T'] - 0.2655577) < 1e-5, f"Mismatch in RCV T: {res.estimate['RCV T']}"
    assert res.p_value < 1e-15, f"Mismatch in p-value: {res.p_value}"
    print('ipcor_rcv_test matches the reference.')
    return res

res_rcv = timed('ipcor_rcv_test', run_ipcor_rcv_test)


	Improved projection correlation RCV asymptotic test

data:  x and y
Z = 13.278, p-value = < 2.2e-16
alternative hypothesis: true correlation is greater than 0
sample estimates:
       RCV T
   0.2655577

ipcor_rcv_test matches the reference.
  [timed] ipcor_rcv_test: 1.6772 s


## Timing Summary

Execution time for every tested procedure (wall-clock seconds, `time.perf_counter`).

In [8]:
print("\n" + "=" * 60)
print("TIMING SUMMARY")
print("=" * 60)
total = 0.0
for name, elapsed in timing_results:
    print(f"{name:<32s} {elapsed:>10.4f} s")
    total += elapsed
print("-" * 60)
print(f"{'TOTAL':<32s} {total:>10.4f} s")
print("=" * 60 + "\n")

if 'res_pcor' in globals() and 'pcor_val' in globals():
    diff = abs(res_pcor.estimate['PC'] - pcor_val)
    print(f"Cross-check |pcor_test.PC - pcor()| = {diff:.2e}")

if 'res_ipcor' in globals() and 'ipcor_val' in globals():
    diff = abs(res_ipcor.estimate['IPC'] - ipcor_val)
    print(f"Cross-check |ipcor_test.IPC - ipcor()| = {diff:.2e}")


TIMING SUMMARY
pcor_test                            0.8563 s
pcor (pcor_ipcor)                    0.1063 s
ipcor_test                           0.0346 s
ipcor (pcor_ipcor)                   0.0046 s
ipcor_rcv_test                       1.6772 s
------------------------------------------------------------
TOTAL                                2.6789 s

Cross-check |pcor_test.PC - pcor()| = 1.04e-10
Cross-check |ipcor_test.IPC - ipcor()| = 0.00e+00


## Summary

If all cells finish without an `AssertionError`, the computed results match the reference within the specified tolerances. The `Compute device` line shows whether this Colab session used CUDA. Timing results are printed above.